# ERU Schedule Generator — Spring 2026
**Egyptian Russian University | Faculty of Management, Economics & Business Technology**  
**Business Technology Department | Course: ARI301 — Introduction to Artificial Intelligence**

---

## Algorithm Overview

| Phase | Method | Purpose | Time |
|-------|--------|---------|------|
| **Phase 1** | CSP — Min-Conflicts Local Search | Find a valid schedule with **0 hard constraint violations** | ~0.03s |
| **Phase 2** | Genetic Algorithm (GA) | Optimize **psychological quality** (soft constraints) from ~62 to ~85/100 | ~18s |

### Hard Constraints (must have 0 violations)
- HC1: No instructor double-booked (includes co-lecturers)
- HC2: No room double-booked
- HC3: No student group double-booked
- HC4: Sessions use valid time blocks (no break spanning)
- HC5: Saturday only for STA103 lecture
- HC6: Room capacity sufficient
- HC7: Labs need computer/lab rooms

### Soft Constraints (weighted 0–100 score)
| Priority | Constraint | Weight |
|----------|-----------|--------|
| (a) | No >2 consecutive lectures | 5.0 |
| (d) | Even spread across week | 4.0 |
| (e) | Social hour (slot 4 free) | 3.0 |
| (b) | Day off per group | 2.0 |
| (f) | Light boundary days (Sun/Thu) | 1.5 |
| (c) | No early+late same day | 1.0 |

---
## Cell 1 — Imports & Dependencies

In [ ]:
import time
import random
import os
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Set
from enum import Enum
from collections import defaultdict
from copy import deepcopy

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False
    print("openpyxl not installed — Excel export disabled. Run: pip install openpyxl")

---
## Cell 2 — Configuration Constants

In [ ]:
# ============================================================
# TIME STRUCTURE
# ============================================================
DAYS = {
    0: "Sunday",
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Saturday",
}

DAY_NAMES = list(DAYS.values())

WORKING_DAYS = [0, 1, 2, 3, 4]          # Sunday-Thursday (normal)
SATURDAY = 5                             # Exception only for STA103

SLOTS = {
    1: ("09:00", "09:50"),
    2: ("09:50", "10:40"),
    # --- BREAK 10:40 - 11:30 ---
    3: ("11:30", "12:20"),
    4: ("12:20", "13:10"),
    5: ("13:10", "14:00"),
    6: ("14:00", "14:50"),
}

BREAK_START = "10:40"
BREAK_END = "11:30"
BREAK_DURATION_MIN = 50
SLOTS_PER_DAY = 6

# Valid consecutive slot blocks (no break spanning)
LECTURE_BLOCKS = [(1, 2), (3, 4), (4, 5), (5, 6)]    # 2 consecutive slots
LAB_BLOCKS = [(1, 2), (3, 4), (4, 5), (5, 6)]        # 2 consecutive slots (1h40m practical)
SECTION_SLOTS = [1, 2, 3, 4, 5, 6]                    # Any single slot

# ============================================================
# SESSION TYPES
# ============================================================
LECTURE = "Lecture"
LAB = "Lab"
SECTION = "Section"

SESSION_DURATIONS = {
    LECTURE: 2,   # 2 slots = 100 min (1:40)
    LAB: 2,       # 2 slots = 100 min (1:40)
    SECTION: 1,   # 1 slot  = 50 min  (0:50)
}

# ============================================================
# ROOM CONFIGURATION
# ============================================================
ROOM_TYPES = {
    "hall":         {"prefix": "Hall", "capacity_range": (100, 150)},
    "classroom_L":  {"prefix": "Room", "capacity_range": (60, 80)},
    "classroom_S":  {"prefix": "Room", "capacity_range": (40, 60)},
    "computer_lab": {"prefix": "Lab",  "capacity_range": (35, 45)},
    "basement_lab": {"prefix": "BLab", "capacity_range": (30, 40)},
}

# ============================================================
# SOFT CONSTRAINT WEIGHTS (ranked by user priority)
# ============================================================
WEIGHT_CONSECUTIVE_LECTURES = 5.0    # (a) No >2 heavy lectures back-to-back
WEIGHT_EVEN_SPREAD = 4.0             # (d) Spread courses evenly across week
WEIGHT_SOCIAL_HOUR = 3.0             # (e) Common free midday slot
WEIGHT_DAY_OFF = 2.0                 # (b) At least 1 full day off per group
WEIGHT_LIGHT_ENDS = 1.5              # (f) Light days at week boundaries
WEIGHT_EARLY_LATE = 1.0              # (c) Avoid first+last slot same day

# ============================================================
# GENETIC ALGORITHM PARAMETERS
# ============================================================
GA_POPULATION_SIZE = 50
GA_GENERATIONS = 200
GA_ELITE_RATIO = 0.10
GA_CROSSOVER_RATE = 0.85
GA_MUTATION_RATE = 0.15
GA_TOURNAMENT_SIZE = 5
GA_STAGNATION_LIMIT = 50

# ============================================================
# LEVEL & MAJOR DEFINITIONS
# ============================================================
LEVELS = [1, 2, 3, 4]
MAJORS_L3 = ["BA", "MIS", "MKI", "FINTECH"]
MAJORS_L4 = ["BA", "MIS", "MKI", "DBFT"]
GROUP_TYPE_NEWCOMERS = "NEWCOMERS"
GROUP_TYPE_REGULAR = "REGULAR"

print("Configuration loaded.")

---
## Cell 3 — Data Models

In [ ]:
class SessionType(Enum):
    LECTURE = LECTURE
    LAB = LAB
    SECTION = SECTION


@dataclass
class Room:
    code: str
    capacity: int
    room_type: str
    has_computers: bool = False

    def __hash__(self):
        return hash(self.code)

    def __eq__(self, other):
        return isinstance(other, Room) and self.code == other.code


@dataclass
class Course:
    code: str
    name: str
    level: int
    group_type: str
    lecture_instructor: Optional[str] = None
    lab_instructors: List[str] = field(default_factory=list)
    section_instructors: List[str] = field(default_factory=list)
    has_lecture: bool = True
    has_lab: bool = False
    has_section: bool = False
    is_common: bool = False
    shared_with: List[str] = field(default_factory=list)
    lecture_group_id: Optional[str] = None
    saturday_only: bool = False
    co_lecturers: List[str] = field(default_factory=list)

    def __hash__(self):
        return hash((self.code, self.level, self.group_type))

    def __eq__(self, other):
        return (isinstance(other, Course) and self.code == other.code
                and self.level == other.level and self.group_type == other.group_type)


@dataclass
class Session:
    session_id: int
    course: Course
    session_type: SessionType
    instructor: str
    duration_slots: int
    target_groups: List[str]
    is_shared_lecture: bool = False

    def __hash__(self):
        return hash(self.session_id)

    def __eq__(self, other):
        return isinstance(other, Session) and self.session_id == other.session_id


@dataclass
class TimeBlock:
    day: int
    slots: Tuple[int, ...]

    @property
    def start_slot(self):
        return self.slots[0]

    @property
    def end_slot(self):
        return self.slots[-1]

    @property
    def day_name(self):
        return DAYS.get(self.day, "Unknown")

    @property
    def time_str(self):
        start = SLOTS[self.start_slot][0]
        end = SLOTS[self.end_slot][1]
        return f"{start}-{end}"

    def overlaps(self, other: 'TimeBlock') -> bool:
        if self.day != other.day:
            return False
        return bool(set(self.slots) & set(other.slots))

    def __hash__(self):
        return hash((self.day, self.slots))

    def __eq__(self, other):
        return (isinstance(other, TimeBlock) and
                self.day == other.day and self.slots == other.slots)


@dataclass
class Assignment:
    session: Session
    time_block: TimeBlock
    room: Room


class Schedule:
    def __init__(self):
        self.assignments: List[Assignment] = []
        self._instructor_map = {}
        self._room_map = {}
        self._group_map = {}

    def add_assignment(self, assignment: Assignment):
        self.assignments.append(assignment)
        s = assignment.session
        tb = assignment.time_block
        inst = s.instructor
        if inst not in self._instructor_map:
            self._instructor_map[inst] = []
        self._instructor_map[inst].append((tb, s.session_id))
        rc = assignment.room.code
        if rc not in self._room_map:
            self._room_map[rc] = []
        self._room_map[rc].append((tb, s.session_id))
        for g in s.target_groups:
            if g not in self._group_map:
                self._group_map[g] = []
            self._group_map[g].append((tb, s.session_id))

    def remove_last(self):
        if not self.assignments:
            return
        a = self.assignments.pop()
        s = a.session
        self._instructor_map[s.instructor] = [
            x for x in self._instructor_map[s.instructor] if x[1] != s.session_id]
        self._room_map[a.room.code] = [
            x for x in self._room_map[a.room.code] if x[1] != s.session_id]
        for g in s.target_groups:
            self._group_map[g] = [
                x for x in self._group_map[g] if x[1] != s.session_id]

    def check_instructor_conflict(self, instructor: str, tb: TimeBlock) -> bool:
        for existing_tb, _ in self._instructor_map.get(instructor, []):
            if existing_tb.overlaps(tb):
                return True
        return False

    def check_room_conflict(self, room_code: str, tb: TimeBlock) -> bool:
        for existing_tb, _ in self._room_map.get(room_code, []):
            if existing_tb.overlaps(tb):
                return True
        return False

    def check_group_conflict(self, groups: List[str], tb: TimeBlock) -> bool:
        for g in groups:
            for existing_tb, _ in self._group_map.get(g, []):
                if existing_tb.overlaps(tb):
                    return True
        return False

    def get_group_schedule(self, group_key: str) -> List[Assignment]:
        session_ids = {sid for _, sid in self._group_map.get(group_key, [])}
        return [a for a in self.assignments if a.session.session_id in session_ids]

    def get_all_groups(self) -> List[str]:
        return sorted(self._group_map.keys())

    def copy(self) -> 'Schedule':
        new_schedule = Schedule()
        for a in self.assignments:
            new_schedule.add_assignment(Assignment(a.session, a.time_block, a.room))
        return new_schedule

    def get_group_days(self, group_key: str) -> dict:
        days = {}
        for tb, sid in self._group_map.get(group_key, []):
            if tb.day not in days:
                days[tb.day] = []
            assignment = next(a for a in self.assignments if a.session.session_id == sid)
            days[tb.day].append(assignment)
        for d in days:
            days[d].sort(key=lambda a: a.time_block.start_slot)
        return days


print("Data models loaded.")

---
## Cell 4 — Data Loader (57 Courses, 24 Rooms, Session Engine)

In [ ]:
# ============================================================
# ROOM DEFINITIONS
# ============================================================
def load_rooms() -> List[Room]:
    return [
        Room("101", 120, "hall"), Room("103", 100, "hall"),
        Room("104", 100, "hall"), Room("105", 100, "hall"),
        Room("201", 80, "classroom_L"), Room("203", 70, "classroom_L"),
        Room("204", 70, "classroom_L"), Room("205", 60, "classroom_L"),
        Room("209", 60, "classroom_L"), Room("210", 60, "classroom_L"),
        Room("211", 60, "classroom_L"),
        Room("301", 55, "classroom_S"), Room("303", 50, "classroom_S"),
        Room("304", 50, "classroom_S"), Room("311", 45, "classroom_S"),
        Room("313", 45, "classroom_S"),
        Room("G01", 40, "computer_lab", has_computers=True),
        Room("G03", 35, "computer_lab", has_computers=True),
        Room("G09", 40, "computer_lab", has_computers=True),
        Room("G12", 35, "computer_lab", has_computers=True),
        Room("B01", 50, "basement_lab"), Room("B09", 35, "basement_lab"),
        Room("B10", 35, "basement_lab"), Room("B11", 35, "basement_lab"),
    ]


# ============================================================
# COURSE DEFINITIONS — ALL LEVELS
# ============================================================

def _newcomer_courses() -> List[Course]:
    return [
        Course("IST103", "Fundamentals of Information Systems", 1, "NEWCOMERS",
               lecture_instructor="Dr. Reham", has_lab=True, lab_instructors=["T.A Paula"]),
        Course("MTH103", "Mathematics", 1, "NEWCOMERS",
               lecture_instructor="Dr. Wael", has_section=True, section_instructors=["T.A Ahmed Ibrahim"]),
        Course("ECO101", "Introduction to Microeconomics", 1, "NEWCOMERS",
               lecture_instructor="Dr. Hossam", has_section=True, section_instructors=["T.A Shrouk"]),
        Course("MGT101", "Introduction to Management", 1, "NEWCOMERS",
               lecture_instructor="Dr. Mona"),
        Course("HM003", "English Language 1", 1, "NEWCOMERS",
               lecture_instructor="Dr. Amany"),
        Course("COS101", "Introduction to Computer Science", 1, "NEWCOMERS",
               lecture_instructor="Dr. Maged", has_lab=True, lab_instructors=["T.A Asmaa"]),
    ]


def _l1_regular_courses() -> List[Course]:
    return [
        Course("STA103", "Statistics and Probability 1", 1, "REGULAR",
               lecture_instructor="Dr. Nada", has_section=True,
               section_instructors=["T.A Salem"], saturday_only=True),
        Course("IST104", "System Analysis and Design 1", 1, "REGULAR",
               lecture_instructor="Dr. Ghada", has_lab=True,
               lab_instructors=["T.A NADA", "T.A OLA"],
               is_common=True, lecture_group_id="IST104_L1"),
        Course("COS102", "Introduction to Programming", 1, "REGULAR",
               lecture_instructor="Dr. Marwa Elsayed", has_lab=True,
               lab_instructors=["T.A Mona", "T.A Safaa"],
               is_common=True, lecture_group_id="COS102_L1"),
        Course("ECO102", "Introduction to Macroeconomics", 1, "REGULAR",
               lecture_instructor="Dr. Saad", has_section=True,
               section_instructors=["T.A M. Gamal"],
               is_common=True, lecture_group_id="ECO102_L1"),
        Course("ACC125", "Financial Accounting", 1, "REGULAR",
               lecture_instructor="Dr. Marwa", has_section=True,
               section_instructors=["T.A Amr", "T.A Nouf"],
               is_common=True, lecture_group_id="ACC125_L1"),
        Course("HM004", "English Language 2", 1, "REGULAR",
               lecture_instructor="Dr. Amany",
               is_common=True, lecture_group_id="HM004_L1"),
    ]


def _l2_courses() -> List[Course]:
    return [
        Course("IST207", "Database System 2", 2, "L2",
               lecture_instructor="Dr. Ghada", has_lab=True,
               lab_instructors=["T.A Mennatullah"],
               is_common=True, lecture_group_id="IST207_L2"),
        Course("BUA201", "Introduction to Business Analytics", 2, "L2",
               lecture_instructor="Dr. Rowaida", has_lab=True,
               lab_instructors=["T.A Salem"],
               is_common=True, lecture_group_id="BUA201_L2"),
        Course("ACC226", "Fundamentals of Managerial Accounting", 2, "L2",
               lecture_instructor="Dr. Mohsen", has_section=True,
               section_instructors=["T.A Amr", "T.A Nouf"],
               is_common=True, lecture_group_id="ACC226_L2"),
        Course("HM005", "Scientific Thinking", 2, "L2",
               lecture_instructor="Dr. Haidy",
               is_common=True, lecture_group_id="HM005_L2"),
        Course("FIN201", "Principles of Finance and Investment", 2, "L2",
               lecture_instructor="Dr. Osama", has_section=True,
               section_instructors=["T.A Sabreen"],
               is_common=True, lecture_group_id="FIN201_L2"),
        Course("MGT102", "Introduction to Marketing", 2, "L2",
               lecture_instructor="Dr. Sara",
               is_common=True, lecture_group_id="MGT102_L2"),
        Course("HM002", "Russian Language 2", 2, "L2",
               lecture_instructor="Dr. Halema",
               is_common=True, lecture_group_id="HM002_L2"),
    ]


def _l3_common_courses() -> List[dict]:
    return [
        {"code": "BUA302", "name": "Marketing Analytics",
         "lecture_instructor": "Dr. Mona",
         "lab_instructors": ["T.A Mona", "T.A Abdelrahman"], "has_lab": True,
         "shared_majors": ["BA", "MIS", "MKI"], "lecture_group_id": "BUA302_L3"},
        {"code": "STA306", "name": "Operation Research",
         "lecture_instructor": "Dr. Nada",
         "lab_instructors": ["T.A Mariem"], "has_lab": True,
         "shared_majors": ["BA", "MIS", "MKI"], "lecture_group_id": "STA306_L3"},
        {"code": "BIT303", "name": "Networking and Telecommunications",
         "lecture_instructor": "Dr. Ayat",
         "lab_instructors": ["T.A Ola", "T.A Abdelfatah"], "has_lab": True,
         "shared_majors": ["BA", "MKI", "FINTECH"], "lecture_group_id": "BIT303_L3L4"},
        {"code": "ARI301", "name": "Introduction to Artificial Intelligence",
         "lecture_instructor": "Dr. Shady",
         "lab_instructors": ["T.A Abdelrahman"], "has_lab": True,
         "shared_majors": ["BA", "MKI"], "lecture_group_id": "ARI301_L3"},
        {"code": "ECO327", "name": "International Economics / Foreign Trade",
         "lecture_instructor": "Dr. Hossam",
         "section_instructors": ["Abdelkader"], "has_section": True,
         "shared_majors": ["MIS", "MKI"], "lecture_group_id": "ECO327_L3"},
    ]


def _l3_ba_courses() -> List[Course]:
    return [
        Course("BUA304", "Predictive Analytics", 3, "BA",
               lecture_instructor="Dr. Rowaida", has_lab=True, lab_instructors=["T.A Mariem"]),
        Course("BUA303", "Business Intelligence", 3, "BA",
               lecture_instructor="Dr. Shady", has_lab=True, lab_instructors=["T.A Nouran"],
               is_common=True, lecture_group_id="BUA303_L3L4", shared_with=["L4_MIS"]),
    ]


def _l3_mis_courses() -> List[Course]:
    return [
        Course("FIN410", "Investment Portfolio Management", 3, "MIS",
               lecture_instructor="Dr. Sohair", has_section=True, section_instructors=["T.A HANIN"]),
        Course("MGT203", "Introduction to Human Resource Management", 3, "MIS",
               lecture_instructor="Dr. Dina"),
        Course("IST311", "Web Based Application", 3, "MIS",
               lecture_instructor="Dr. Reham", has_lab=True, lab_instructors=["T.A Paula"]),
        Course("MIS303", "Decision Making and Information Systems", 3, "MIS",
               lecture_instructor="Dr. Ghada", has_lab=True, lab_instructors=["T.A Abdelfatah", "T.A Ola"]),
    ]


def _l3_mki_courses() -> List[Course]:
    return [
        Course("IST303", "Advanced Topics in Information Systems", 3, "MKI",
               lecture_instructor="Dr. Tarek", has_lab=True, lab_instructors=["T.A Abdelfatah"]),
    ]


def _l3_fintech_courses() -> List[Course]:
    return [
        Course("ECO310", "Econometrics", 3, "FINTECH",
               lecture_instructor="Dr. Engy", has_section=True, section_instructors=["T.A Idwa"]),
        Course("DBF301", "Principles of Digital Banking and Fintech", 3, "FINTECH",
               lecture_instructor="Dr. Engy", has_section=True, section_instructors=["T.A Idwa"]),
        Course("COS320", "Distributed System", 3, "FINTECH",
               lecture_instructor="Dr. Tarek", has_lab=True, lab_instructors=["T.A Safaa"]),
        Course("DBF302", "Valuation of Companies and Cash Flow", 3, "FINTECH",
               lecture_instructor="Dr. Osama", has_section=True, section_instructors=["T.A Sabreen"]),
        Course("MGT307", "Business Ethics and CSR", 3, "FINTECH",
               lecture_instructor="Dr. Sara"),
    ]


def _l4_ba_courses() -> List[Course]:
    return [
        Course("BUA409", "Business Forecasting", 4, "BA",
               lecture_instructor="Dr. Rowaida", has_lab=True, lab_instructors=["T.A Nouran"]),
        Course("IST309", "Information Retrieval and Web Search", 4, "BA",
               lecture_instructor="Dr. Ayat", has_lab=True, lab_instructors=["T.A Mariem"]),
        Course("DAS410", "Foundation of Big Data", 4, "BA",
               lecture_instructor="Dr. Yasser", has_lab=True, lab_instructors=["T.A Abdelrahman"]),
        Course("BIT406", "Data Security", 4, "BA",
               lecture_instructor="Dr. Yasser", has_lab=True, lab_instructors=["T.A Nada"]),
        Course("BUA425", "Graduation Project 2", 4, "BA",
               lecture_instructor="Dr. Rowaida", co_lecturers=["Dr. Shady"]),
    ]


def _l4_mis_courses() -> List[Course]:
    return [
        Course("BUA303", "Business Intelligence", 4, "MIS",
               lecture_instructor="Dr. Shady", has_lab=True, lab_instructors=["T.A Nouran"],
               is_common=True, lecture_group_id="BUA303_L3L4", shared_with=["L3_BA"]),
        Course("MIS408", "Accounting Information Systems", 4, "MIS",
               lecture_instructor="Dr. Hend", has_lab=True, lab_instructors=["T.A Paula", "T.A Fadeel"]),
        Course("BIT303", "Networking and Telecommunications", 4, "MIS",
               lecture_instructor="Dr. Ayat", has_lab=True,
               lab_instructors=["T.A Ola", "T.A Abdelfatah"],
               is_common=True, lecture_group_id="BIT303_L3L4",
               shared_with=["L3_BA", "L3_MKI", "L3_FINTECH"]),
        Course("MGT416", "Strategic Management", 4, "MIS",
               lecture_instructor="Dr. Dina"),
        Course("MIS425", "Graduation Project 2", 4, "MIS",
               lecture_instructor="Dr. Ghada", co_lecturers=["Dr. Reham"]),
    ]


def _l4_mki_courses() -> List[Course]:
    return [
        Course("MKI403", "Customer Analytics", 4, "MKI",
               lecture_instructor="Dr. Mona", has_lab=True, lab_instructors=["T.A Nouran Amr"]),
        Course("MKT408", "International Marketing", 4, "MKI",
               lecture_instructor="Dr. Mona", has_section=True, section_instructors=["T.A Alaa"]),
        Course("MKT406", "Brand Management", 4, "MKI",
               lecture_instructor="Dr. Sara Elmeneawy", has_section=True, section_instructors=["T.A Alaa Nagib"]),
        Course("MGT434", "Marketing Strategy", 4, "MKI",
               lecture_instructor="Dr. Pancee", has_section=True, section_instructors=["T.A Nouran"]),
        Course("MKI402", "Graduation Project 2", 4, "MKI",
               lecture_instructor="Dr. Yasser"),
        Course("MGT415", "Graduation Project", 4, "MKI",
               lecture_instructor="Dr. Mona"),
    ]


def _l4_dbft_courses() -> List[Course]:
    return [
        Course("DBF406", "Financial Derivatives", 4, "DBFT",
               lecture_instructor="Dr. Ehab", has_section=True, section_instructors=["T.A Sabreen"]),
        Course("FIN314", "Financial Risk Management", 4, "DBFT",
               lecture_instructor="Dr. Ehab", has_section=True, section_instructors=["T.A Sabreen"]),
        Course("COS422", "Blockchain and Crypto Assets", 4, "DBFT",
               lecture_instructor="Dr. Marwa Elsayed", has_lab=True, lab_instructors=["T.A Nada"]),
        Course("COS424", "Database Security", 4, "DBFT",
               lecture_instructor="Dr. Maged", has_lab=True, lab_instructors=["T.A Menna"]),
        Course("DBF407", "Graduation Project 2", 4, "DBFT",
               lecture_instructor="Dr. Engy"),
    ]


# ============================================================
# SESSION GENERATION ENGINE
# ============================================================

class DataLoader:
    def __init__(self):
        self.rooms = load_rooms()
        self.courses: List[Course] = []
        self.sessions: List[Session] = []
        self.group_keys: List[str] = []
        self._session_counter = 0
        self._lecture_group_sessions: Dict[str, Session] = {}
        self.l1_regular_count = 0
        self.l2_group_count = 0
        self.student_counts: Dict[str, int] = {}

    def _next_id(self) -> int:
        self._session_counter += 1
        return self._session_counter

    def configure_groups(self, l1_regular_count: int, l2_group_count: int,
                         student_counts: Dict[str, int]):
        self.l1_regular_count = l1_regular_count
        self.l2_group_count = l2_group_count
        self.student_counts = student_counts

    def load_all(self):
        self.courses, self.sessions = [], []
        self._session_counter = 0
        self._lecture_group_sessions = {}
        self.group_keys = []
        self._load_l1_newcomers()
        self._load_l1_regular()
        self._load_l2()
        self._load_l3()
        self._load_l4()

    def _make_group_keys(self, level: int, group_type: str, count: int = 1) -> List[str]:
        if level == 1 and group_type == "NEWCOMERS":
            keys = ["L1_NEW"]
        elif level == 1 and group_type == "REGULAR":
            keys = [f"L1_{chr(65+i)}" for i in range(count)]
        elif level == 2:
            keys = [f"L2_{chr(65+i)}" for i in range(count)]
        else:
            keys = [f"L{level}_{group_type}"]
        for k in keys:
            if k not in self.group_keys:
                self.group_keys.append(k)
        return keys

    def _create_lecture_session(self, course: Course, target_groups: List[str]) -> Session:
        if course.lecture_group_id and course.lecture_group_id in self._lecture_group_sessions:
            existing = self._lecture_group_sessions[course.lecture_group_id]
            for g in target_groups:
                if g not in existing.target_groups:
                    existing.target_groups.append(g)
            return existing
        session = Session(
            session_id=self._next_id(), course=course,
            session_type=SessionType.LECTURE, instructor=course.lecture_instructor,
            duration_slots=SESSION_DURATIONS[LECTURE],
            target_groups=list(target_groups), is_shared_lecture=course.is_common)
        self.sessions.append(session)
        if course.lecture_group_id:
            self._lecture_group_sessions[course.lecture_group_id] = session
        return session

    def _create_lab_sections(self, course: Course, group_keys: List[str]):
        if course.has_lab and course.lab_instructors:
            ta_count = len(course.lab_instructors)
            for i, gk in enumerate(group_keys):
                ta = course.lab_instructors[i % ta_count]
                session = Session(
                    session_id=self._next_id(), course=course,
                    session_type=SessionType.LAB, instructor=ta,
                    duration_slots=SESSION_DURATIONS[LAB], target_groups=[gk])
                self.sessions.append(session)
        if course.has_section and course.section_instructors:
            ta_count = len(course.section_instructors)
            for i, gk in enumerate(group_keys):
                ta = course.section_instructors[i % ta_count]
                session = Session(
                    session_id=self._next_id(), course=course,
                    session_type=SessionType.SECTION, instructor=ta,
                    duration_slots=SESSION_DURATIONS[SECTION], target_groups=[gk])
                self.sessions.append(session)

    def _load_l1_newcomers(self):
        gk = self._make_group_keys(1, "NEWCOMERS")
        for course in _newcomer_courses():
            self.courses.append(course)
            if course.has_lecture: self._create_lecture_session(course, gk)
            self._create_lab_sections(course, gk)

    def _load_l1_regular(self):
        gk = self._make_group_keys(1, "REGULAR", self.l1_regular_count)
        for course in _l1_regular_courses():
            self.courses.append(course)
            if course.has_lecture: self._create_lecture_session(course, gk)
            self._create_lab_sections(course, gk)

    def _load_l2(self):
        gk = self._make_group_keys(2, "L2", self.l2_group_count)
        for course in _l2_courses():
            self.courses.append(course)
            if course.has_lecture: self._create_lecture_session(course, gk)
            self._create_lab_sections(course, gk)

    def _load_l3(self):
        major_groups = {}
        for m in MAJORS_L3:
            major_groups[m] = self._make_group_keys(3, m)
        for cdef in _l3_common_courses():
            all_tg = []
            for m in cdef["shared_majors"]: all_tg.extend(major_groups[m])
            course = Course(
                code=cdef["code"], name=cdef["name"], level=3, group_type="L3_COMMON",
                lecture_instructor=cdef["lecture_instructor"],
                has_lab=cdef.get("has_lab", False), lab_instructors=cdef.get("lab_instructors", []),
                has_section=cdef.get("has_section", False), section_instructors=cdef.get("section_instructors", []),
                is_common=True, lecture_group_id=cdef["lecture_group_id"],
                shared_with=[f"L3_{m}" for m in cdef["shared_majors"]])
            self.courses.append(course)
            if course.has_lecture: self._create_lecture_session(course, all_tg)
            for m in cdef["shared_majors"]: self._create_lab_sections(course, major_groups[m])
        for courses_fn, major in [(_l3_ba_courses, "BA"), (_l3_mis_courses, "MIS"),
                                   (_l3_mki_courses, "MKI"), (_l3_fintech_courses, "FINTECH")]:
            for course in courses_fn():
                self.courses.append(course)
                gk = major_groups[major]
                if course.has_lecture:
                    if course.lecture_group_id:
                        cross_groups = list(gk)
                        for sw in course.shared_with:
                            sw_key = sw.replace("L3_", "").replace("L4_", "")
                            sw_level = 3 if sw.startswith("L3") else 4
                            cross_groups.extend(self._make_group_keys(sw_level, sw_key))
                        self._create_lecture_session(course, cross_groups)
                    else:
                        self._create_lecture_session(course, gk)
                self._create_lab_sections(course, gk)

    def _load_l4(self):
        major_groups = {}
        for m in MAJORS_L4:
            major_groups[m] = self._make_group_keys(4, m)
        for courses_fn, major in [(_l4_ba_courses, "BA"), (_l4_mis_courses, "MIS"),
                                   (_l4_mki_courses, "MKI"), (_l4_dbft_courses, "DBFT")]:
            for course in courses_fn():
                self.courses.append(course)
                gk = major_groups[major]
                if course.has_lecture:
                    if course.lecture_group_id:
                        cross_groups = list(gk)
                        for sw in course.shared_with:
                            sw_key = sw.replace("L3_", "").replace("L4_", "")
                            sw_level = 3 if sw.startswith("L3") else 4
                            cross_groups.extend(self._make_group_keys(sw_level, sw_key))
                        self._create_lecture_session(course, cross_groups)
                    else:
                        self._create_lecture_session(course, gk)
                self._create_lab_sections(course, gk)

    def get_rooms_for_session(self, session: Session, student_count: int) -> List[Room]:
        suitable = []
        for room in self.rooms:
            if room.capacity >= student_count:
                if session.session_type == SessionType.LAB and not room.has_computers:
                    if room.room_type not in ("computer_lab", "basement_lab"): continue
                suitable.append(room)
        if not suitable:
            suitable = sorted(self.rooms, key=lambda r: r.capacity, reverse=True)[:5]
        return suitable

    def get_valid_time_blocks(self, session: Session) -> List[TimeBlock]:
        blocks = []
        if session.course.saturday_only and session.session_type == SessionType.LECTURE:
            days = [SATURDAY]
        else:
            days = WORKING_DAYS
        if session.session_type == SessionType.LECTURE:
            for day in days:
                for block in LECTURE_BLOCKS: blocks.append(TimeBlock(day=day, slots=block))
        elif session.session_type == SessionType.LAB:
            for day in days:
                for block in LAB_BLOCKS: blocks.append(TimeBlock(day=day, slots=tuple(block)))
        else:
            for day in days:
                for slot in SECTION_SLOTS: blocks.append(TimeBlock(day=day, slots=(slot,)))
        return blocks

    def summary(self) -> str:
        lectures = sum(1 for s in self.sessions if s.session_type == SessionType.LECTURE)
        labs = sum(1 for s in self.sessions if s.session_type == SessionType.LAB)
        sections = sum(1 for s in self.sessions if s.session_type == SessionType.SECTION)
        return (f"Courses: {len(self.courses)} | Sessions: {len(self.sessions)} "
                f"(Lec:{lectures} Lab:{labs} Sec:{sections}) | "
                f"Rooms: {len(self.rooms)} | Groups: {len(self.group_keys)}\n"
                f"Groups: {', '.join(self.group_keys)}")


print("Data loader ready.")

---
## Cell 5 — Conflict Manager (Hard Constraint Validation)

In [ ]:
class ConflictManager:
    def __init__(self, schedule: Schedule):
        self.schedule = schedule

    def instructor_conflict(self, session: Session, time_block: TimeBlock) -> bool:
        if self.schedule.check_instructor_conflict(session.instructor, time_block): return True
        for co in getattr(session.course, 'co_lecturers', []):
            if self.schedule.check_instructor_conflict(co, time_block): return True
        return False

    def room_conflict(self, room: Room, time_block: TimeBlock) -> bool:
        return self.schedule.check_room_conflict(room.code, time_block)

    def group_conflict(self, session: Session, time_block: TimeBlock) -> bool:
        return self.schedule.check_group_conflict(session.target_groups, time_block)

    def block_validity(self, session: Session, time_block: TimeBlock) -> bool:
        valid_blocks = ([tuple(b) for b in LECTURE_BLOCKS] if session.session_type == SessionType.LECTURE
                        else [tuple(b) for b in LAB_BLOCKS] if session.session_type == SessionType.LAB
                        else None)
        if valid_blocks is not None: return tuple(time_block.slots) not in valid_blocks
        return time_block.slots[0] not in SECTION_SLOTS

    def saturday_constraint(self, session: Session, time_block: TimeBlock) -> bool:
        is_sat_lecture = (session.course.saturday_only and session.session_type == SessionType.LECTURE)
        if time_block.day == SATURDAY: return not is_sat_lecture
        if is_sat_lecture: return time_block.day != SATURDAY
        return False

    def room_capacity(self, room: Room, student_count: int) -> bool:
        return room.capacity < student_count

    def lab_room_type(self, session: Session, room: Room) -> bool:
        if session.session_type == SessionType.LAB:
            return room.room_type not in ("computer_lab", "basement_lab")
        return False

    def has_conflict(self, session, time_block, room, student_count=0) -> bool:
        return (self.instructor_conflict(session, time_block) or
                self.room_conflict(room, time_block) or
                self.group_conflict(session, time_block) or
                self.block_validity(session, time_block) or
                self.saturday_constraint(session, time_block) or
                (student_count > 0 and self.room_capacity(room, student_count)) or
                self.lab_room_type(session, room))

    @staticmethod
    def validate_schedule(schedule: Schedule) -> List[str]:
        violations = []
        assignments = schedule.assignments
        for i in range(len(assignments)):
            a1 = assignments[i]
            for j in range(i + 1, len(assignments)):
                a2 = assignments[j]
                if not a1.time_block.overlaps(a2.time_block): continue
                all_inst_1 = {a1.session.instructor} | set(getattr(a1.session.course, 'co_lecturers', []))
                all_inst_2 = {a2.session.instructor} | set(getattr(a2.session.course, 'co_lecturers', []))
                if all_inst_1 & all_inst_2:
                    violations.append(f"INSTRUCTOR CONFLICT: {all_inst_1 & all_inst_2} on {a1.time_block.day_name} {a1.time_block.time_str} ({a1.session.course.code} vs {a2.session.course.code})")
                if a1.room.code == a2.room.code:
                    violations.append(f"ROOM CONFLICT: Room {a1.room.code} on {a1.time_block.day_name} {a1.time_block.time_str} ({a1.session.course.code} vs {a2.session.course.code})")
                shared_groups = set(a1.session.target_groups) & set(a2.session.target_groups)
                if shared_groups:
                    violations.append(f"GROUP CONFLICT: {shared_groups} on {a1.time_block.day_name} {a1.time_block.time_str} ({a1.session.course.code} vs {a2.session.course.code})")
        for a in assignments:
            if a.time_block.day == SATURDAY:
                if not (a.session.course.saturday_only and a.session.session_type == SessionType.LECTURE):
                    violations.append(f"SATURDAY VIOLATION: {a.session.course.code} {a.session.session_type.value} on Saturday")
            if a.session.session_type == SessionType.LAB:
                if a.room.room_type not in ("computer_lab", "basement_lab"):
                    violations.append(f"LAB ROOM VIOLATION: {a.session.course.code} lab in non-lab room {a.room.code}")
        return violations


print("Conflict manager ready.")

---
## Cell 6 — CSP Solver (Min-Conflicts Local Search + Greedy Room Assignment)

In [ ]:
class CSPSolver:
    def __init__(self, data_loader: DataLoader, student_counts: Dict[str, int]):
        self.data_loader = data_loader
        self.student_counts = student_counts
        self.nodes_explored = 0

    def _get_student_count(self, session: Session) -> int:
        return sum(self.student_counts.get(g, 40) for g in session.target_groups)

    def _build_conflict_graph(self, sessions: List[Session]):
        self._conflicts = defaultdict(set)
        self._session_map = {s.session_id: s for s in sessions}
        for i, s1 in enumerate(sessions):
            for j in range(i + 1, len(sessions)):
                s2 = sessions[j]
                if self._sessions_conflict(s1, s2):
                    self._conflicts[s1.session_id].add(s2.session_id)
                    self._conflicts[s2.session_id].add(s1.session_id)

    def _sessions_conflict(self, s1: Session, s2: Session) -> bool:
        if s1.instructor == s2.instructor: return True
        for co in getattr(s1.course, 'co_lecturers', []):
            if co == s2.instructor: return True
        for co in getattr(s2.course, 'co_lecturers', []):
            if co == s1.instructor: return True
        if set(s1.target_groups) & set(s2.target_groups): return True
        return False

    def _get_time_domain(self, session: Session) -> List[TimeBlock]:
        blocks = self.data_loader.get_valid_time_blocks(session)
        is_sat_lecture = (session.course.saturday_only and session.session_type == SessionType.LECTURE)
        if is_sat_lecture: return [tb for tb in blocks if tb.day == SATURDAY]
        return [tb for tb in blocks if tb.day != SATURDAY]

    def _count_violations(self, sid, tb, assignments):
        count = 0
        for other_sid in self._conflicts[sid]:
            if other_sid in assignments and assignments[other_sid].overlaps(tb): count += 1
        return count

    def _solve_times(self, sessions, max_restarts=100):
        self._build_conflict_graph(sessions)
        self.nodes_explored = 0
        domains = {s.session_id: self._get_time_domain(s) for s in sessions}
        all_sids = [s.session_id for s in sessions]
        for restart in range(max_restarts):
            assignments = {sid: random.choice(domains[sid]) for sid in all_sids}
            violation_count = {sid: self._count_violations(sid, assignments[sid], assignments) for sid in all_sids}
            for step in range(len(sessions) * 300):
                self.nodes_explored += 1
                conflicted = [sid for sid in all_sids if violation_count[sid] > 0]
                if not conflicted:
                    print(f"    Solution found: restart {restart}, step {step} (nodes={self.nodes_explored})")
                    return assignments
                sid = random.choice(conflicted)
                old_tb = assignments[sid]
                min_v, best_tbs = float('inf'), []
                for tb in domains[sid]:
                    v = self._count_violations(sid, tb, assignments)
                    if v < min_v: min_v, best_tbs = v, [tb]
                    elif v == min_v: best_tbs.append(tb)
                new_tb = random.choice(best_tbs)
                if new_tb != old_tb:
                    for o in self._conflicts[sid]:
                        if o in assignments and old_tb.overlaps(assignments[o]): violation_count[o] -= 1
                    assignments[sid] = new_tb
                    violation_count[sid] = min_v
                    for o in self._conflicts[sid]:
                        if o in assignments and new_tb.overlaps(assignments[o]): violation_count[o] += 1
            if restart % 10 == 0:
                print(f"    Restart {restart}: {sum(1 for s in all_sids if violation_count[s]>0)} still conflicting")
        return None

    def _assign_rooms(self, sessions, time_assignments):
        schedule = Schedule()
        session_map = {s.session_id: s for s in sessions}
        rooms = self.data_loader.rooms
        sorted_sids = sorted(time_assignments.keys(), key=lambda sid: (
            0 if session_map[sid].session_type == SessionType.LAB else
            1 if session_map[sid].session_type == SessionType.LECTURE else 2,
            -self._get_student_count(session_map[sid])))
        room_assignments = {}
        for sid in sorted_sids:
            session = session_map[sid]
            tb = time_assignments[sid]
            sc = self._get_student_count(session)
            used = {room_assignments[o].code for o, otb in time_assignments.items()
                    if o != sid and tb.overlaps(otb) and o in room_assignments}
            suitable = self.data_loader.get_rooms_for_session(session, sc)
            available = [r for r in suitable if r.code not in used]
            if not available: available = [r for r in rooms if r.code not in used and r.capacity >= min(sc, 30)]
            if not available: available = [r for r in rooms if r.code not in used]
            if not available: return None
            available.sort(key=lambda r: abs(r.capacity - sc))
            room_assignments[sid] = available[0]
        for sid in sorted_sids:
            schedule.add_assignment(Assignment(session_map[sid], time_assignments[sid], room_assignments[sid]))
        return schedule

    def solve(self):
        sessions = self.data_loader.sessions
        lec = sum(1 for s in sessions if s.session_type == SessionType.LECTURE)
        lab = sum(1 for s in sessions if s.session_type == SessionType.LAB)
        sec = sum(1 for s in sessions if s.session_type == SessionType.SECTION)
        print(f"  [CSP] Sessions: {len(sessions)} (Lec:{lec} Lab:{lab} Sec:{sec}) | Rooms: {len(self.data_loader.rooms)}")
        total_start = time.time()
        print(f"  [CSP] === PASS 1: Time Block Assignment ===")
        p1_start = time.time()
        time_assignments = self._solve_times(sessions)
        p1_time = time.time() - p1_start
        if time_assignments is None:
            print("  [CSP] FAILED: Could not assign time blocks"); return None
        print(f"  [CSP] Pass 1: {len(time_assignments)} assigned in {p1_time:.3f}s")
        print(f"  [CSP] === PASS 2: Room Assignment ===")
        p2_start = time.time()
        schedule = self._assign_rooms(sessions, time_assignments)
        p2_time = time.time() - p2_start
        if schedule is None:
            print("  [CSP] FAILED: Could not assign rooms"); return None
        total_time = time.time() - total_start
        violations = ConflictManager.validate_schedule(schedule)
        print(f"  [CSP] Total: {total_time:.3f}s (time:{p1_time:.3f}s + rooms:{p2_time:.3f}s)")
        print(f"  [CSP] Assignments: {len(schedule.assignments)}/{len(sessions)} | Violations: {len(violations)} | Nodes: {self.nodes_explored}")
        if violations:
            for v in violations[:5]: print(f"    {v}")
        return schedule


print("CSP solver ready.")

---
## Cell 7 — Soft Constraint Evaluator (6 Scoring Functions)

In [ ]:
class Evaluator:
    def __init__(self, schedule: Schedule):
        self.schedule = schedule
        self._group_days_cache: Dict = {}

    def _get_group_days(self, group_key: str) -> dict:
        if group_key not in self._group_days_cache:
            self._group_days_cache[group_key] = self.schedule.get_group_days(group_key)
        return self._group_days_cache[group_key]

    def evaluate(self) -> Dict:
        groups = self.schedule.get_all_groups()
        if not groups: return {"total": 0, "breakdown": {}, "weights": {}}
        self._group_days_cache = {}
        scores = {
            "consecutive_lectures": self._score_consecutive_lectures(groups),
            "even_spread": self._score_even_spread(groups),
            "social_hour": self._score_social_hour(groups),
            "day_off": self._score_day_off(groups),
            "light_boundaries": self._score_light_boundaries(groups),
            "early_late": self._score_early_late(groups),
        }
        weights = {
            "consecutive_lectures": WEIGHT_CONSECUTIVE_LECTURES,
            "even_spread": WEIGHT_EVEN_SPREAD, "social_hour": WEIGHT_SOCIAL_HOUR,
            "day_off": WEIGHT_DAY_OFF, "light_boundaries": WEIGHT_LIGHT_ENDS,
            "early_late": WEIGHT_EARLY_LATE,
        }
        total_w = sum(weights.values())
        total = sum(scores[k] * weights[k] for k in scores) / total_w if total_w > 0 else 0
        return {"total": round(total, 1), "breakdown": {k: round(v, 1) for k, v in scores.items()}, "weights": weights}

    def _score_consecutive_lectures(self, groups):
        total_days, violations = 0, 0
        for g in groups:
            for day, assignments in self._get_group_days(g).items():
                total_days += 1
                lectures = sorted([(a.time_block.start_slot, a.time_block.end_slot)
                                   for a in assignments if a.session.session_type == SessionType.LECTURE])
                if len(lectures) >= 3:
                    consec, max_c = 1, 1
                    for i in range(1, len(lectures)):
                        if lectures[i][0] - lectures[i-1][1] <= 1: consec += 1; max_c = max(max_c, consec)
                        else: consec = 1
                    if max_c > 2: violations += 1
        return max(0, 100 * (1 - violations / total_days)) if total_days else 100.0

    def _score_even_spread(self, groups):
        total = 0
        for g in groups:
            ds = self._get_group_days(g)
            counts = [len(ds.get(d, [])) for d in WORKING_DAYS]
            active = [c for c in counts if c > 0]
            if not active: total += 100; continue
            mean = sum(active) / len(active)
            if mean == 0: total += 100; continue
            var = sum((x - mean)**2 for x in active) / len(active)
            total += max(0, 100 * (1 - (var**0.5) / mean))
        return total / len(groups) if groups else 100

    def _score_social_hour(self, groups):
        total, day_count = 0, 0
        for day in WORKING_DAYS:
            day_count += 1
            free = 0
            for g in groups:
                slots_used = set()
                for a in self._get_group_days(g).get(day, []): slots_used.update(a.time_block.slots)
                if 4 not in slots_used: free += 1
            total += (free / len(groups) * 100) if groups else 0
        return total / day_count if day_count else 100

    def _score_day_off(self, groups):
        count = 0
        for g in groups:
            active = set(self._get_group_days(g).keys()) - {SATURDAY}
            if set(WORKING_DAYS) - active: count += 1
        return (count / len(groups) * 100) if groups else 100

    def _score_light_boundaries(self, groups):
        total = 0
        for g in groups:
            ds = self._get_group_days(g)
            boundary = sum(len(ds.get(d, [])) for d in [0, 4])
            mid = sum(len(ds.get(d, [])) for d in [1, 2, 3])
            avg_b = boundary / 2 if boundary else 0
            avg_m = mid / 3 if mid else 0
            if avg_m == 0 and avg_b == 0: total += 100
            elif avg_b <= avg_m: total += 100
            else: total += max(0, 100 * (2 - (avg_b / avg_m if avg_m else 2)))
        return total / len(groups) if groups else 100

    def _score_early_late(self, groups):
        total_days, violations = 0, 0
        for g in groups:
            for day, assignments in self._get_group_days(g).items():
                total_days += 1
                slots_used = set()
                for a in assignments: slots_used.update(a.time_block.slots)
                if 1 in slots_used and 6 in slots_used: violations += 1
        return max(0, 100 * (1 - violations / total_days)) if total_days else 100.0

    def report(self) -> str:
        result = self.evaluate()
        labels = {"consecutive_lectures": "(a) No >2 consecutive lectures",
                  "even_spread": "(d) Even spread across week",
                  "social_hour": "(e) Social hour overlap",
                  "day_off": "(b) Day off per group",
                  "light_boundaries": "(f) Light boundary days",
                  "early_late": "(c) No early+late same day"}
        lines = [f"\n{'='*60}", f"  SCHEDULE QUALITY: {result['total']}/100", f"{'='*60}"]
        for key, label in labels.items():
            s = result["breakdown"].get(key, 0)
            w = result["weights"].get(key, 0)
            bar = '#' * int(s/5) + '-' * (20 - int(s/5))
            lines.append(f"  {label:40s} {bar} {s:5.1f} (x{w})")
        lines.append(f"{'='*60}")
        return '\n'.join(lines)


print("Evaluator ready.")

---
## Cell 8 — Genetic Algorithm Optimizer

In [ ]:
class Chromosome:
    def __init__(self):
        self.genes: Dict[int, Tuple[int, tuple, str]] = {}
        self.fitness: float = 0.0

    def to_schedule(self, sessions_map, rooms_map) -> Schedule:
        schedule = Schedule()
        for sid, (day, slots, room_code) in self.genes.items():
            schedule.add_assignment(Assignment(sessions_map[sid], TimeBlock(day=day, slots=slots), rooms_map[room_code]))
        return schedule

    @staticmethod
    def from_schedule(schedule: Schedule) -> 'Chromosome':
        c = Chromosome()
        for a in schedule.assignments:
            c.genes[a.session.session_id] = (a.time_block.day, a.time_block.slots, a.room.code)
        return c

    def copy(self) -> 'Chromosome':
        c = Chromosome()
        c.genes = dict(self.genes)
        c.fitness = self.fitness
        return c


class GeneticOptimizer:
    def __init__(self, data_loader: DataLoader, initial_schedule: Schedule, student_counts: Dict[str, int]):
        self.data_loader = data_loader
        self.initial_schedule = initial_schedule
        self.student_counts = student_counts
        self.sessions_map = {s.session_id: s for s in data_loader.sessions}
        self.rooms_map = {r.code: r for r in data_loader.rooms}
        self.valid_domains: Dict[int, List[Tuple[int, tuple, str]]] = {}
        self._precompute_domains()
        self.population: List[Chromosome] = []
        self.best_chromosome = None
        self.best_fitness = 0.0
        self.generation_history: List[float] = []

    def _precompute_domains(self):
        for session in self.data_loader.sessions:
            tbs = self.data_loader.get_valid_time_blocks(session)
            max_s = sum(self.student_counts.get(g, 40) for g in session.target_groups)
            rooms = self.data_loader.get_rooms_for_session(session, max_s)
            self.valid_domains[session.session_id] = [(tb.day, tb.slots, r.code) for tb in tbs for r in rooms]

    def _fast_validate(self, chrom):
        slot_instr, slot_room, slot_group = {}, {}, {}
        for sid, (day, slots, room_code) in chrom.genes.items():
            session = self.sessions_map[sid]
            is_sat_lec = (session.course.saturday_only and session.session_type == SessionType.LECTURE)
            if day == SATURDAY and not is_sat_lec: return False
            if is_sat_lec and day != SATURDAY: return False
            all_inst = [session.instructor] + list(getattr(session.course, 'co_lecturers', []))
            for slot in slots:
                key = (day, slot)
                for inst in all_inst:
                    if (key, inst) in slot_instr: return False
                    slot_instr[(key, inst)] = sid
                if (key, room_code) in slot_room: return False
                slot_room[(key, room_code)] = sid
                for g in session.target_groups:
                    if (key, g) in slot_group: return False
                    slot_group[(key, g)] = sid
        return True

    def _evaluate_chromosome(self, chrom):
        if not self._fast_validate(chrom): return -1.0
        schedule = chrom.to_schedule(self.sessions_map, self.rooms_map)
        return Evaluator(schedule).evaluate()["total"]

    def _is_valid_assignment(self, chrom, session_id, new_gene):
        day, slots, room_code = new_gene
        session = self.sessions_map[session_id]
        tb = TimeBlock(day=day, slots=slots)
        is_sat_lec = (session.course.saturday_only and session.session_type == SessionType.LECTURE)
        if day == SATURDAY and not is_sat_lec: return False
        if is_sat_lec and day != SATURDAY: return False
        all_inst = {session.instructor} | set(getattr(session.course, 'co_lecturers', []))
        for o_sid, (o_day, o_slots, o_room) in chrom.genes.items():
            if o_sid == session_id: continue
            o_tb = TimeBlock(day=o_day, slots=o_slots)
            if not tb.overlaps(o_tb): continue
            o_sess = self.sessions_map[o_sid]
            o_inst = {o_sess.instructor} | set(getattr(o_sess.course, 'co_lecturers', []))
            if all_inst & o_inst: return False
            if room_code == o_room: return False
            if set(session.target_groups) & set(o_sess.target_groups): return False
        return True

    def _init_population(self):
        self.population = []
        seed = Chromosome.from_schedule(self.initial_schedule)
        seed.fitness = self._evaluate_chromosome(seed)
        self.population.append(seed)
        self.best_chromosome = seed.copy()
        self.best_fitness = seed.fitness
        print(f"  [GA] Seed fitness: {seed.fitness:.1f}/100")
        attempts = 0
        while len(self.population) < GA_POPULATION_SIZE and attempts < GA_POPULATION_SIZE * 20:
            attempts += 1
            v = seed.copy()
            for _ in range(random.randint(3, min(15, len(v.genes)//2))): self._mutate(v)
            v.fitness = self._evaluate_chromosome(v)
            if v.fitness >= 0: self.population.append(v)
        print(f"  [GA] Population: {len(self.population)} ({attempts} attempts)")

    def _tournament_select(self):
        return max(random.sample(self.population, min(GA_TOURNAMENT_SIZE, len(self.population))), key=lambda c: c.fitness)

    def _crossover(self, p1, p2):
        child = Chromosome()
        days_from_p1 = set(random.sample(WORKING_DAYS, random.randint(1, len(WORKING_DAYS)-1)))
        for sid in p1.genes:
            child.genes[sid] = p1.genes[sid] if p1.genes[sid][0] in days_from_p1 else p2.genes.get(sid, p1.genes[sid])
        for sid in p2.genes:
            if sid not in child.genes: child.genes[sid] = p2.genes[sid]
        return child

    def _mutate(self, chrom):
        if not chrom.genes: return False
        sid = random.choice(list(chrom.genes.keys()))
        domain = self.valid_domains.get(sid, [])
        if not domain: return False
        random.shuffle(domain)
        for new_gene in domain[:20]:
            if self._is_valid_assignment(chrom, sid, new_gene):
                chrom.genes[sid] = new_gene; return True
        return False

    def optimize(self) -> Schedule:
        print(f"\n  [GA] Pop: {GA_POPULATION_SIZE}, Gen: {GA_GENERATIONS}")
        ga_start = time.time()
        self._init_population()
        if not self.population: return self.initial_schedule
        stagnation = 0
        for gen in range(GA_GENERATIONS):
            new_pop = []
            self.population.sort(key=lambda c: c.fitness, reverse=True)
            elite_n = max(1, int(len(self.population) * GA_ELITE_RATIO))
            elites = [c.copy() for c in self.population[:elite_n]]
            new_pop.extend(elites)
            while len(new_pop) < len(self.population):
                p1, p2 = self._tournament_select(), self._tournament_select()
                child = self._crossover(p1, p2) if random.random() < GA_CROSSOVER_RATE else p1.copy()
                if random.random() < GA_MUTATION_RATE: self._mutate(child)
                child.fitness = self._evaluate_chromosome(child)
                if child.fitness >= 0: new_pop.append(child)
                else:
                    fb = random.choice(elites).copy(); self._mutate(fb)
                    fb.fitness = self._evaluate_chromosome(fb)
                    if fb.fitness >= 0: new_pop.append(fb)
            self.population = new_pop[:GA_POPULATION_SIZE]
            gb = max(self.population, key=lambda c: c.fitness)
            self.generation_history.append(gb.fitness)
            if gb.fitness > self.best_fitness:
                self.best_fitness, self.best_chromosome, stagnation = gb.fitness, gb.copy(), 0
            else: stagnation += 1
            if gen % 50 == 0 or gen == GA_GENERATIONS - 1:
                avg = sum(c.fitness for c in self.population) / len(self.population)
                print(f"    Gen {gen:4d} | Best: {self.best_fitness:5.1f} | Avg: {avg:5.1f} | Stag: {stagnation}")
            if stagnation >= GA_STAGNATION_LIMIT:
                print(f"  [GA] Converged at gen {gen}"); break
        print(f"  [GA] Done in {time.time()-ga_start:.2f}s | Best: {self.best_fitness:.1f}/100")
        return self.best_chromosome.to_schedule(self.sessions_map, self.rooms_map)


print("Genetic optimizer ready.")

---
## Cell 9 — Formatter (Console Grid + Detailed View + Excel Export)

In [ ]:
class Formatter:
    def __init__(self, schedule: Schedule):
        self.schedule = schedule

    def print_group_schedule(self, group_key: str):
        assignments = self.schedule.get_group_schedule(group_key)
        if not assignments: print(f"  No sessions for {group_key}"); return
        grid = {d: {} for d in WORKING_DAYS + [SATURDAY]}
        for a in assignments:
            for slot in a.time_block.slots: grid[a.time_block.day][slot] = a
        active_days = [d for d in WORKING_DAYS if grid[d]]
        if grid[SATURDAY]: active_days.append(SATURDAY)
        if not active_days: return
        label = group_key.replace('_', ' - ').replace('L', 'Level ')
        print(f"\n{'='*90}\n  SCHEDULE: {label}\n{'='*90}")
        cw = 16
        hdr = f"  {'Slot':>10}" + ''.join(f" | {DAYS[d]:^{cw}}" for d in active_days)
        print(hdr); print(f"  {'-'*10}" + ('-+-' + '-'*cw) * len(active_days))
        for slot in [1, 2]:
            row = f"  {SLOTS[slot][0]:>10}"
            for d in active_days: row += f" | {self._fmt(grid[d].get(slot), slot):{cw}}"
            print(row)
        bs = f"{BREAK_START}-{BREAK_END}"
        print(f"  {'-'*10}" + ('-+-' + '-'*cw) * len(active_days))
        print(f"  {'BREAK':>10}" + ''.join(f" | {bs:^{cw}}" for _ in active_days))
        print(f"  {'-'*10}" + ('-+-' + '-'*cw) * len(active_days))
        for slot in [3, 4, 5, 6]:
            row = f"  {SLOTS[slot][0]:>10}"
            for d in active_days: row += f" | {self._fmt(grid[d].get(slot), slot):{cw}}"
            print(row)
        print(f"{'='*90}")

    def _fmt(self, a, slot):
        if a is None: return '---'
        if a.time_block.start_slot != slot: return '  (cont.)'
        abbr = {'Lecture': 'Lec', 'Lab': 'Lab', 'Section': 'Sec'}
        return f"{a.session.course.code}({abbr.get(a.session.session_type.value, '?')})"

    def print_group_detail(self, group_key: str):
        assignments = self.schedule.get_group_schedule(group_key)
        if not assignments: return
        label = group_key.replace('_', ' - ').replace('L', 'Level ')
        print(f"\n{'-'*70}\n  DETAILED: {label}\n{'-'*70}")
        by_day = {}
        for a in assignments: by_day.setdefault(a.time_block.day, []).append(a)
        for day in sorted(by_day):
            print(f"\n  {DAYS[day]}:")
            for a in sorted(by_day[day], key=lambda x: x.time_block.start_slot):
                print(f"    {a.time_block.time_str:14s} {a.session.course.code:10s} "
                      f"{a.session.course.name:40s} {a.session.session_type.value:8s} "
                      f"{a.session.instructor:20s} [{a.room.code}]")

    def print_all_schedules(self):
        groups = self.schedule.get_all_groups()
        levels = {}
        for g in groups: levels.setdefault(g.split('_')[0], []).append(g)
        for level in sorted(levels):
            print(f"\n\n{'#'*90}\n  {level.replace('L','LEVEL ')} SCHEDULES\n{'#'*90}")
            for gk in sorted(levels[level]):
                self.print_group_schedule(gk)
                self.print_group_detail(gk)

    def print_conflict_report(self):
        v = ConflictManager.validate_schedule(self.schedule)
        print(f"\n{'='*60}\n  CONFLICT REPORT\n{'='*60}")
        if not v: print(f"  Status: ALL CLEAR (0 violations)")
        else:
            print(f"  VIOLATIONS: {len(v)}")
            for x in v[:20]: print(f"    - {x}")
        print(f"{'='*60}")

    def print_summary(self):
        a_list = self.schedule.assignments
        groups = self.schedule.get_all_groups()
        lec = sum(1 for a in a_list if a.session.session_type == SessionType.LECTURE)
        lab = sum(1 for a in a_list if a.session.session_type == SessionType.LAB)
        sec = sum(1 for a in a_list if a.session.session_type == SessionType.SECTION)
        rooms_used = set(a.room.code for a in a_list)
        print(f"\n{'='*60}\n  SUMMARY\n{'='*60}")
        print(f"  Sessions: {len(a_list)} | Groups: {len(groups)} | Lec:{lec} Lab:{lab} Sec:{sec}")
        print(f"  Rooms: {len(rooms_used)} | Instructors: {len(set(a.session.instructor for a in a_list))}")
        for g in groups:
            ds = self.schedule.get_group_days(g)
            act = [DAYS[d] for d in sorted(ds.keys())]
            free = set(WORKING_DAYS) - set(ds.keys())
            fn = [DAYS[d] for d in sorted(free)] if free else ['None']
            print(f"    {g:12s} | {sum(len(v) for v in ds.values()):2d} sessions | Free: {', '.join(fn)}")
        print(f"{'='*60}")

    def export_to_excel(self, filepath: str):
        if not HAS_OPENPYXL: print("  [EXPORT] openpyxl not installed. Skipping."); return
        wb = Workbook(); wb.remove(wb.active)
        groups = self.schedule.get_all_groups()
        colors = {SessionType.LECTURE: 'B8D4E3', SessionType.LAB: 'D5E8D4', SessionType.SECTION: 'FFF2CC'}
        hfill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
        hfont = Font(color='FFFFFF', bold=True, size=10)
        border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
        for gk in groups:
            ws = wb.create_sheet(title=gk[:31])
            ws.merge_cells('A1:H1')
            ws['A1'].value = f'ERU Schedule Spring 2026 - {gk}'
            ws['A1'].font = Font(bold=True, size=14)
            ws['A1'].alignment = Alignment(horizontal='center')
            headers = ['Time/Day'] + [DAYS[d] for d in WORKING_DAYS]
            for col, h in enumerate(headers, 1):
                c = ws.cell(row=3, column=col, value=h)
                c.fill, c.font, c.alignment, c.border = hfill, hfont, Alignment(horizontal='center'), border
            asgn = self.schedule.get_group_schedule(gk)
            grid = {d: {} for d in WORKING_DAYS + [SATURDAY]}
            for a in asgn:
                for slot in a.time_block.slots: grid[a.time_block.day][slot] = a
            row = 4
            for slot in [1, 2]:
                ws.cell(row=row, column=1, value=f"{SLOTS[slot][0]}-{SLOTS[slot][1]}").border = border
                for col, day in enumerate(WORKING_DAYS, 2):
                    cell = ws.cell(row=row, column=col); cell.border = border
                    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
                    a = grid[day].get(slot)
                    if a and a.time_block.start_slot == slot:
                        cell.value = f"{a.session.course.code}\n{a.session.session_type.value}\n{a.session.instructor}\n[{a.room.code}]"
                        fc = colors.get(a.session.session_type, 'FFFFFF')
                        cell.fill = PatternFill(start_color=fc, end_color=fc, fill_type='solid')
                    elif a: cell.value = '(cont.)'
                row += 1
            ws.cell(row=row, column=1, value=f'BREAK {BREAK_START}-{BREAK_END}').font = Font(bold=True)
            for col in range(1, len(headers)+1):
                ws.cell(row=row, column=col).fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
                ws.cell(row=row, column=col).border = border
            row += 1
            for slot in [3, 4, 5, 6]:
                ws.cell(row=row, column=1, value=f"{SLOTS[slot][0]}-{SLOTS[slot][1]}").border = border
                for col, day in enumerate(WORKING_DAYS, 2):
                    cell = ws.cell(row=row, column=col); cell.border = border
                    cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
                    a = grid[day].get(slot)
                    if a and a.time_block.start_slot == slot:
                        cell.value = f"{a.session.course.code}\n{a.session.session_type.value}\n{a.session.instructor}\n[{a.room.code}]"
                        fc = colors.get(a.session.session_type, 'FFFFFF')
                        cell.fill = PatternFill(start_color=fc, end_color=fc, fill_type='solid')
                    elif a: cell.value = '(cont.)'
                row += 1
            ws.column_dimensions['A'].width = 18
            for col in range(2, len(headers)+1): ws.column_dimensions[chr(64+col)].width = 22
            for r in range(4, row): ws.row_dimensions[r].height = 60
        try:
            wb.save(filepath); print(f"  [EXPORT] Saved: {filepath}")
        except PermissionError:
            alt = filepath.replace('.xlsx', '_new.xlsx')
            wb.save(alt); print(f"  [EXPORT] File locked, saved to: {alt}")


print("Formatter ready.")

---
## Cell 10 — User Input & Configuration

In [ ]:
def get_int_input(prompt, default=None, min_val=1):
    while True:
        try:
            suffix = f" [{default}]" if default else ""
            val = input(f"  {prompt}{suffix}: ").strip()
            if val == '' and default is not None: return default
            val = int(val)
            if val < min_val: print(f"    Please enter >= {min_val}"); continue
            return val
        except ValueError: print("    Enter a valid number.")


print("="*60)
print("  ERU SCHEDULE GENERATOR -- Spring 2026")
print("  Faculty of Management, Economics & Business Technology")
print("  Algorithm: CSP + Min-Conflicts + Genetic Algorithm")
print("="*60)

student_counts = {}

print("\n  [LEVEL 1]")
l1_regular_count = get_int_input("Number of L1 regular groups (A, B, ...)", default=2)
l1_new_students = get_int_input("Students in Newcomers group", default=40)
l1_reg_students = get_int_input("Students per regular L1 group", default=40)
student_counts["L1_NEW"] = l1_new_students
for i in range(l1_regular_count): student_counts[f"L1_{chr(65+i)}"] = l1_reg_students

print("\n  [LEVEL 2]")
l2_group_count = get_int_input("Number of L2 groups", default=2)
l2_students = get_int_input("Students per L2 group", default=35)
for i in range(l2_group_count): student_counts[f"L2_{chr(65+i)}"] = l2_students

print("\n  [LEVEL 3]")
for major in MAJORS_L3:
    student_counts[f"L3_{major}"] = get_int_input(f"Students in L3 {major}", default=30)

print("\n  [LEVEL 4]")
for major in MAJORS_L4:
    student_counts[f"L4_{major}"] = get_int_input(f"Students in L4 {major}", default=25)

print("\nConfiguration complete.")

---
## Cell 11 — Phase 0: Load Data

In [ ]:
loader = DataLoader()
loader.configure_groups(l1_regular_count, l2_group_count, student_counts)
loader.load_all()
print(loader.summary())

---
## Cell 12 — Phase 1: CSP Solver (Find Valid Schedule)

In [ ]:
csp_start = time.time()
solver = CSPSolver(loader, student_counts)
csp_schedule = solver.solve()
csp_time = time.time() - csp_start

if csp_schedule is None:
    raise RuntimeError("CSP solver could not find a valid schedule! Try reducing group counts.")

csp_eval = Evaluator(csp_schedule)
csp_result = csp_eval.evaluate()
csp_violations = ConflictManager.validate_schedule(csp_schedule)

print(f"\n  CSP Score: {csp_result['total']}/100 | Time: {csp_time:.3f}s | Violations: {len(csp_violations)}")

---
## Cell 13 — Phase 2: Genetic Algorithm (Optimize Quality)

In [ ]:
ga_start = time.time()
optimizer = GeneticOptimizer(loader, csp_schedule, student_counts)
optimized_schedule = optimizer.optimize()
ga_time = time.time() - ga_start

ga_eval = Evaluator(optimized_schedule)
ga_result = ga_eval.evaluate()
ga_violations = ConflictManager.validate_schedule(optimized_schedule)

final_schedule = optimized_schedule if len(ga_violations) == 0 else csp_schedule
if len(ga_violations) > 0:
    print("  WARNING: GA schedule has violations, using CSP solution")

---
## Cell 14 — Results Comparison (CSP vs GA)

In [ ]:
print(f"\n  {'Metric':<35} {'CSP Only':>10} {'CSP + GA':>10} {'Change':>10}")
print(f"  {'-'*35} {'-'*10} {'-'*10} {'-'*10}")
diff = ga_result['total'] - csp_result['total']
print(f"  {'Overall Score':<35} {csp_result['total']:>9.1f} {ga_result['total']:>9.1f} {'+' if diff>=0 else ''}{diff:>9.1f}")

labels = {"consecutive_lectures": "(a) Consecutive lectures",
          "even_spread": "(d) Even spread", "social_hour": "(e) Social hour",
          "day_off": "(b) Day off", "light_boundaries": "(f) Light boundaries",
          "early_late": "(c) Early+late avoidance"}
for key in csp_result['breakdown']:
    cv, gv = csp_result['breakdown'][key], ga_result['breakdown'][key]
    d = gv - cv
    print(f"  {labels.get(key,key):<35} {cv:>9.1f} {gv:>9.1f} {'+' if d>=0 else ''}{d:>9.1f}")

print(f"\n  CSP time: {csp_time:.3f}s | GA time: {ga_time:.2f}s | Total: {csp_time+ga_time:.2f}s")
print(f"  Hard violations (final): {len(ga_violations)}")

---
## Cell 15 — Quality Report & Validation

In [ ]:
final_eval = Evaluator(final_schedule)
print(final_eval.report())

formatter = Formatter(final_schedule)
formatter.print_conflict_report()
formatter.print_summary()

---
## Cell 16 — Display All Group Schedules

In [ ]:
formatter.print_all_schedules()

---
## Cell 17 — Export to Excel

In [ ]:
export_path = os.path.join(os.getcwd(), "ERU_Schedule_Output.xlsx")
formatter.export_to_excel(export_path)
print("\nSchedule generation complete!")